# 00 Testing

Notebook for quick tests and experiments.

In [6]:
"""
check_restatements.py

Diagnostic script to understand how Sharadar handles restatements
in the SF1 ARQ data and what impact they have on NCFO R2 calculations.

Usage:
    python check_restatements.py --sf1 path/to/SHARADAR_SF1.parquet

If you have the SF1 CSV instead of parquet:
    python check_restatements.py --sf1 path/to/SHARADAR_SF1.csv

Tickers investigated:
    GE    - major insurance reserve restatements 2017-2019
    UAA   - revenue recognition restatement 2019
    AAPL  - clean baseline, should show no restatements
    V     - clean baseline, high quality compounder
    DVN   - Devon Energy, cyclical oil & gas (replacement for CHK if no data)
"""

import argparse
import sys
from pathlib import Path

# Notebook: ensure project root on path so config and SF1 path resolve
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy import stats


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def parse_args():
    p = argparse.ArgumentParser(description="Sharadar restatement diagnostic")
    p.add_argument("--sf1", required=True,
                   help="Path to SHARADAR_SF1 parquet or CSV file")
    p.add_argument("--tickers", nargs="*",
                   default=["GE", "UAA", "AAPL", "V", "DVN"],
                   help="Tickers to investigate (default: GE UAA AAPL V DVN)")
    p.add_argument("--materiality", type=float, default=0.05,
                   help="Threshold for material restatement (default: 0.05 = 5%%)")
    # When run in Jupyter there is no --sf1 in argv; use project SF1 path from config
    if "--sf1" not in sys.argv:
        import config
        sf1_path = config.DATA_DIR / "SF1.parquet"
        if not sf1_path.exists():
            sf1_path = config.DATA_DIR / "sf1.parquet"
        from types import SimpleNamespace
        return SimpleNamespace(
            sf1=str(sf1_path),
            tickers=["GE", "UAA", "AAPL", "V", "DVN"],
            materiality=0.05,
        )
    return p.parse_args()


# ---------------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------------

def load_sf1(path: str, tickers: list) -> pd.DataFrame:
    """Load SF1 filtered to tickers and ARQ/ARY dimensions."""
    p = Path(path)
    print(f"Loading SF1 from {p} ...")

    if p.suffix == ".parquet":
        df = pd.read_parquet(p)
    elif p.suffix in (".csv", ".gz"):
        df = pd.read_csv(p, low_memory=False)
    else:
        sys.exit(f"Unsupported file type: {p.suffix}. Expected .parquet or .csv")

    # Normalize column names to lowercase
    df.columns = df.columns.str.lower()

    # Filter to tickers and AR dimensions only
    df = df[
        df["ticker"].isin(tickers) &
        df["dimension"].isin(["ARQ", "ARY", "ART"])
    ].copy()

    # Parse dates
    for col in ["datekey", "reportperiod", "calendardate"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    df = df.sort_values(["ticker", "dimension", "reportperiod", "datekey"])
    print(f"  Loaded {len(df):,} rows for {df.ticker.nunique()} tickers\n")
    return df


# ---------------------------------------------------------------------------
# Section 1 – How many versions exist per period?
# ---------------------------------------------------------------------------

SECTION_SEP = "=" * 70

def section(title: str):
    print(f"\n{SECTION_SEP}")
    print(f"  {title}")
    print(SECTION_SEP)


def check_version_counts(sf1: pd.DataFrame, tickers: list):
    """
    For each ticker show how many ARQ rows exist per reportperiod.
    Multiple rows = Sharadar stored amendments / restatements.
    Single row   = only original filing stored.
    """
    section("1. VERSION COUNTS PER REPORTPERIOD (ARQ only)")

    arq = sf1[sf1.dimension == "ARQ"].copy()

    for ticker in tickers:
        t = arq[arq.ticker == ticker]
        if t.empty:
            print(f"\n{ticker}: no ARQ data found")
            continue

        counts = (
            t.groupby("reportperiod")
             .size()
             .reset_index(name="n_versions")
        )
        multi = counts[counts.n_versions > 1]
        total_periods = len(counts)
        restated_periods = len(multi)

        print(f"\n{ticker}:")
        print(f"  Total reportperiods : {total_periods}")
        print(f"  Periods with >1 row : {restated_periods}  "
              f"({'none – single filing only' if restated_periods == 0 else 'restatements/amendments present'})")

        if restated_periods > 0:
            print(f"\n  Restated periods (first 5):")
            for _, row in multi.head(5).iterrows():
                period = row["reportperiod"]
                versions = t[t.reportperiod == period][
                    ["datekey", "ncfo", "revenue", "netinccmn"]
                ].copy()
                versions["datekey"] = versions["datekey"].dt.strftime("%Y-%m-%d")
                print(f"\n    reportperiod = {period.strftime('%Y-%m-%d')}")
                print(versions.to_string(index=False, col_space=14))


# ---------------------------------------------------------------------------
# Section 2 – Magnitude of changes between versions
# ---------------------------------------------------------------------------

def check_restatement_magnitudes(sf1: pd.DataFrame, tickers: list,
                                  materiality: float):
    """
    For periods with multiple ARQ rows compare the first and last filing.
    Flags changes that exceed the materiality threshold.
    """
    section(f"2. RESTATEMENT MAGNITUDES  (materiality threshold = {materiality:.0%})")

    arq = sf1[sf1.dimension == "ARQ"].copy()

    for ticker in tickers:
        t = arq[arq.ticker == ticker]
        if t.empty:
            continue

        records = []
        for period, grp in t.groupby("reportperiod"):
            grp = grp.sort_values("datekey")
            if len(grp) < 2:
                continue  # no restatement

            original  = grp.iloc[0]
            restated  = grp.iloc[-1]

            for metric in ["ncfo", "revenue", "netinccmn"]:
                orig_val = original.get(metric, np.nan)
                rest_val = restated.get(metric, np.nan)

                if pd.isna(orig_val) or pd.isna(rest_val):
                    continue
                if orig_val == 0:
                    continue

                pct_change = (rest_val - orig_val) / abs(orig_val)
                records.append({
                    "period":          period.strftime("%Y-%m-%d"),
                    "metric":          metric,
                    "original":        round(float(orig_val), 0),
                    "restated":        round(float(rest_val), 0),
                    "pct_change":      round(pct_change * 100, 2),
                    "material":        abs(pct_change) >= materiality,
                    "original_date":   original["datekey"].strftime("%Y-%m-%d"),
                    "restatement_date": restated["datekey"].strftime("%Y-%m-%d"),
                })

        print(f"\n{ticker}:")
        if not records:
            print("  No multi-version periods found – no restatements detected")
            continue

        df = pd.DataFrame(records)
        material = df[df.material]
        print(f"  Total amended metric-periods : {len(df)}")
        print(f"  Material (>={materiality:.0%}) : {len(material)}")

        if len(material) > 0:
            print(f"\n  Material restatements:")
            print(material[[
                "period", "metric", "original", "restated",
                "pct_change", "original_date", "restatement_date"
            ]].to_string(index=False, col_space=16))


# ---------------------------------------------------------------------------
# Section 3 – PIT-correct NCFO time series
# ---------------------------------------------------------------------------

def get_pit_ncfo_series(arq: pd.DataFrame, ticker: str,
                         sim_date: pd.Timestamp) -> pd.Series:
    """
    Build the PIT-correct annual NCFO series for a ticker as of sim_date.

    For each reportperiod take the most recently filed ARQ row
    whose datekey <= sim_date.  This is exactly what your ASOF JOIN does.
    Then aggregate to annual by summing four quarters.
    """
    t = arq[
        (arq.ticker == ticker) &
        (arq.datekey <= sim_date)
    ].copy()

    if t.empty:
        return pd.Series(dtype=float)

    # Most recent filing per reportperiod (PIT-correct)
    pit = (
        t.sort_values("datekey")
         .groupby("reportperiod")
         .last()
         .reset_index()
         .sort_values("reportperiod")
    )

    # Aggregate to annual – sum four quarters per calendar year
    pit["year"] = pit["reportperiod"].dt.year
    annual = (
        pit.groupby("year")["ncfo"]
           .sum()
           .reset_index()
           .rename(columns={"ncfo": "ncfo_annual"})
    )

    # Require at least 3 quarters per year to include the year
    quarterly_counts = pit.groupby("year").size().reset_index(name="n_quarters")
    annual = annual.merge(quarterly_counts, on="year")
    annual = annual[annual.n_quarters >= 3]

    return annual.set_index("year")["ncfo_annual"]


def compute_r2_cagr(series: pd.Series):
    """OLS on log(ncfo) vs time index.  Returns (r2, cagr) or (None, None)."""
    s = series.dropna()
    s = s[s > 0]  # log requires positive values

    if len(s) < 4:
        return None, None

    t = np.arange(len(s))
    slope, intercept, r, p_value, std_err = stats.linregress(t, np.log(s))
    return round(r ** 2, 4), round(np.exp(slope) - 1, 4)


def check_r2_pit_impact(sf1: pd.DataFrame, tickers: list):
    """
    Compute NCFO R2 on three simulation dates for each ticker and show
    how restatements shift the metric over time.

    sim_dates chosen to bracket known restatement windows.
    """
    section("3. NCFO R2 ACROSS SIMULATION DATES  (PIT-correct)")

    arq = sf1[sf1.dimension == "ARQ"].copy()

    # Three sim dates: before, during, after typical restatement window
    sim_dates = {
        "2017-06-15": pd.Timestamp("2017-06-15"),
        "2019-06-15": pd.Timestamp("2019-06-15"),
        "2021-06-15": pd.Timestamp("2021-06-15"),
    }

    rows = []
    for ticker in tickers:
        for label, sim_date in sim_dates.items():
            series = get_pit_ncfo_series(arq, ticker, sim_date)
            r2, cagr = compute_r2_cagr(series)
            rows.append({
                "ticker":    ticker,
                "sim_date":  label,
                "n_years":   len(series[series > 0]) if series is not None else 0,
                "r2":        r2,
                "cagr":      f"{cagr:.1%}" if cagr is not None else "N/A",
            })

    df = pd.DataFrame(rows)

    for ticker in tickers:
        sub = df[df.ticker == ticker]
        print(f"\n{ticker}:")
        print(sub[["sim_date", "n_years", "r2", "cagr"]].to_string(index=False))


# ---------------------------------------------------------------------------
# Section 4 – Original vs restated R2 comparison
# ---------------------------------------------------------------------------

def check_r2_original_vs_restated(sf1: pd.DataFrame, tickers: list):
    """
    For each ticker compute R2 using:
      (a) original filing values only  (first datekey per period)
      (b) most recently filed values   (last datekey per period, i.e. MR behavior)

    The difference shows how much restatements affect the quality metric.
    If the difference is large the restatement is material for your model.
    """
    section("4. R2 COMPARISON:  ORIGINAL vs MOST-RECENTLY-RESTATED  (as of 2021-06-15)")
    r2_delta = {}

    arq  = sf1[sf1.dimension == "ARQ"].copy()
    sim  = pd.Timestamp("2021-06-15")

    print(f"\n{'Ticker':<8} {'R2 (original)':<16} {'R2 (restated)':<16} "
          f"{'Delta':<10} {'Note'}")
    print("-" * 65)

    for ticker in tickers:
        t = arq[
            (arq.ticker == ticker) &
            (arq.datekey <= sim)
        ].copy()

        if t.empty:
            print(f"{ticker:<8} {'no data'}")
            continue

        # Original: first filing per period
        original_series_df = (
            t.sort_values("datekey")
             .groupby("reportperiod")
             .first()
             .reset_index()
        )
        original_series_df["year"] = original_series_df["reportperiod"].dt.year
        orig_annual = (
            original_series_df.groupby("year")["ncfo"].sum()
        )

        # Most recent: last filing per period (what MRQ would give)
        restated_series_df = (
            t.sort_values("datekey")
             .groupby("reportperiod")
             .last()
             .reset_index()
        )
        restated_series_df["year"] = restated_series_df["reportperiod"].dt.year
        rest_annual = (
            restated_series_df.groupby("year")["ncfo"].sum()
        )

        r2_orig, _  = compute_r2_cagr(orig_annual)
        r2_rest, _  = compute_r2_cagr(rest_annual)

        if r2_orig is None or r2_rest is None:
            print(f"{ticker:<8} {'insufficient data'}")
            continue

        delta = r2_rest - r2_orig
        r2_delta[ticker] = round(delta, 6)
        note  = ""
        if abs(delta) < 0.01:
            note = "negligible impact"
        elif abs(delta) < 0.05:
            note = "minor impact"
        elif delta < 0:
            note = "*** restatements LOWER R2 (quality overstated in original)"
        else:
            note = "restatements raise R2 (quality understated in original)"

        print(f"{ticker:<8} {r2_orig:<16.4f} {r2_rest:<16.4f} "
              f"{delta:<+10.4f} {note}")
    return r2_delta


# ---------------------------------------------------------------------------
# Section 5 – What your ASOF JOIN actually sees
# ---------------------------------------------------------------------------

def check_asof_behavior(sf1: pd.DataFrame, tickers: list):
    """
    Simulate exactly what your pipeline's ASOF JOIN returns on a series
    of dates for a specific ticker.  Shows how the data visible to the
    model changes as restatements arrive.
    """
    section("5. WHAT YOUR ASOF JOIN SEES ON EACH DATE  (GE example)")

    arq = sf1[sf1.dimension == "ARQ"].copy()

    # Use GE if available, fall back to first ticker
    ticker = "GE" if "GE" in tickers else tickers[0]
    t = arq[arq.ticker == ticker]

    if t.empty:
        print(f"\n{ticker}: no ARQ data")
        return

    # Monthly sim dates across the restatement window
    check_dates = pd.date_range("2017-01-01", "2021-01-01", freq="QS")

    print(f"\n{ticker} – NCFO visible on each sim date via ASOF JOIN:\n")
    print(f"{'sim_date':<14} {'last_filing_date':<18} {'days_stale':<12} "
          f"{'ncfo_ttm':<14} {'r2_10y'}")
    print("-" * 72)

    for sim_date in check_dates:
        # Most recent filing available
        available = t[t.datekey <= sim_date]
        if available.empty:
            continue

        last_filing = available.datekey.max()
        days_stale  = (sim_date - last_filing).days

        # TTM NCFO: sum of last 4 quarters visible on sim_date
        pit = (
            available
            .sort_values("datekey")
            .groupby("reportperiod")
            .last()
            .reset_index()
            .sort_values("reportperiod")
            .tail(4)
        )
        ncfo_ttm = pit["ncfo"].sum() if len(pit) >= 4 else np.nan

        # R2 as of this sim date
        series = get_pit_ncfo_series(arq, ticker, sim_date)
        r2, _  = compute_r2_cagr(series)

        print(f"{sim_date.strftime('%Y-%m-%d'):<14} "
              f"{last_filing.strftime('%Y-%m-%d'):<18} "
              f"{days_stale:<12} "
              f"{ncfo_ttm:<14,.0f} "
              f"{r2 if r2 is not None else 'N/A'}")


# ---------------------------------------------------------------------------
# Section 6 – Summary and recommendations
# ---------------------------------------------------------------------------

def print_summary(tickers: list):
    section("6. SUMMARY AND WHAT TO DO")

    print("""
What this script tells you:

  Section 1  – Whether Sharadar stores multiple ARQ rows per period.
               If you see >1 version per period restatements ARE present
               in the data and your ASOF JOIN handles them correctly.
               If you see only 1 version per period Sharadar stores the
               original filing only (still PIT-correct, just less granular).

  Section 2  – How large the restatements are.  < 5%% = immaterial for R2.
               > 10%% on NCFO = meaningful discontinuity in quality series.

  Section 3  – Whether R2 changes across simulation dates.  A stable R2
               means restatements have little practical impact on your
               quality filter.  A large shift on a specific date means a
               restatement arrived that changed the series meaningfully.

  Section 4  – The difference between using original vs restated values.
               If delta is < 0.02 restatements are not affecting your
               quality ranking materially and you can proceed as-is.
               If delta > 0.05 on multiple tickers consider adding a
               restatement_flag feature to your model.

  Section 5  – Exactly what your model sees on each date.  Use this to
               verify a specific ticker you are concerned about.

Recommended actions based on findings:

  If Sharadar stores only original values (Section 1 shows 1 row/period):
    → Your NCFO R2 is computed on as-originally-reported data.
    → This is PIT-correct but ignores amendments.
    → Add n_material_restatements as a quality flag feature.
    → Source amendment data from SEC EDGAR if high precision needed.

  If Sharadar stores amendments (Section 1 shows >1 row/period):
    → Your ASOF JOIN is already handling this correctly.
    → The PIT series reflects whatever was known on each sim date.
    → Verify Section 4 delta is small for your high-conviction tickers.
    → No further action required for the pipeline.

  Either way:
    → Add a restatement_history_flag as a model feature.
    → Companies with material downward NCFO restatements have lower
      earnings quality regardless of current R2 level.
    → This is free alpha from data you already have.
""")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    args = parse_args()
    tickers = [t.upper() for t in args.tickers]

    sf1 = load_sf1(args.sf1, tickers)

    if sf1.empty:
        sys.exit("No data loaded. Check your file path and ticker list.")

    check_version_counts(sf1, tickers)
    check_restatement_magnitudes(sf1, tickers, args.materiality)
    check_r2_pit_impact(sf1, tickers)
    r2_delta = check_r2_original_vs_restated(sf1, tickers)
    check_asof_behavior(sf1, tickers)
    print_summary(tickers)

    print(f"\n{SECTION_SEP}")
    print("  DONE")
    print(SECTION_SEP)
    return sf1, tickers, r2_delta


if __name__ == "__main__":
    sf1, tickers, r2_delta = main()

Loading SF1 from C:\Users\zachf\regime\data\SF1.parquet ...
  Loaded 1,212 rows for 5 tickers


  1. VERSION COUNTS PER REPORTPERIOD (ARQ only)

GE:
  Total reportperiods : 129
  Periods with >1 row : 2  (restatements/amendments present)

  Restated periods (first 5):

    reportperiod = 2006-09-30
       datekey           ncfo        revenue      netinccmn
    2006-10-31   9491000000.0   4.085600e+10   4964000000.0
    2007-01-19   9491000000.0   4.069300e+10   4867000000.0

    reportperiod = 2016-09-30
       datekey           ncfo        revenue      netinccmn
    2016-11-02   1040000000.0   2.926600e+10   1994000000.0
    2016-11-09   1040000000.0   2.926600e+10   1994000000.0

UAA:
  Total reportperiods : 82
  Periods with >1 row : 1  (restatements/amendments present)

  Restated periods (first 5):

    reportperiod = 2012-12-31
       datekey           ncfo        revenue      netinccmn
    2013-02-25    205773000.0    505863000.0     50132000.0
    2013-02-26    205773000.0    

In [7]:
# Formal assertion tests (run after the diagnostic cell above)
# Requires: sf1, tickers, r2_delta, get_pit_ncfo_series, compute_r2_cagr from previous cell

arq = sf1[sf1.dimension == "ARQ"].copy()

def get_r2(ticker, date):
    """NCFO R2 for ticker as of sim date (PIT-correct)."""
    series = get_pit_ncfo_series(arq, ticker, pd.Timestamp(date))
    r2, _ = compute_r2_cagr(series)
    return r2 if r2 is not None else 0.0

# Restatements are cosmetic - delta should be zero
assert r2_delta.get("GE", 0.0) == 0.0, "GE R2 delta should be zero"
assert r2_delta.get("AAPL", 0.0) == 0.0, "AAPL R2 delta should be zero"

# Quality signal detects GE deterioration correctly
ge_2017 = get_r2("GE", "2017-06-15")
ge_2019 = get_r2("GE", "2019-06-15")
assert ge_2019 < 0.05, f"GE R2 should collapse by 2019, got {ge_2019}"
assert ge_2017 > ge_2019, "GE quality should deteriorate 2017 to 2019"

# AAPL stable high quality
aapl_2017 = get_r2("AAPL", "2017-06-15")
aapl_2021 = get_r2("AAPL", "2021-06-15")
assert aapl_2017 > 0.80, f"AAPL R2 should be high, got {aapl_2017}"
assert abs(aapl_2021 - aapl_2017) < 0.05, "AAPL R2 should be stable"

print("All assertion tests passed.")

All assertion tests passed.


## Verify Sharadar capex sign (AAPL ART)

Before trusting FCF-based metrics in 02_fundamentals we need to confirm whether Sharadar stores **capex** as negative (cash outflow) or positive. FCF = NCFO - |capex|, so:
- If capex is **negative**: use `fcf_recon = ncfo + capex`.
- If capex is **positive**: use `fcf_recon = ncfo - capex`.

Run the cell below to compare `ncfo - capex` and `ncfo + capex` to Sharadar's `fcf` for AAPL ART (2020-2021).

In [ ]:
# Capex sign check: use sf1 from first cell if available, else load SF1 for AAPL
import sys
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if "sf1" not in dir() or sf1 is None or sf1.empty:
    import config
    sf1_path = config.DATA_DIR / "SF1.parquet"
    if not sf1_path.exists():
        sf1_path = config.DATA_DIR / "sf1.parquet"
    sf1 = pd.read_parquet(sf1_path)
    sf1.columns = sf1.columns.str.lower()
    sf1 = sf1[sf1["ticker"].isin(["AAPL"]) & sf1["dimension"].isin(["ARQ", "ARY", "ART"])].copy()
    for col in ["datekey", "reportperiod", "calendardate"]:
        if col in sf1.columns:
            sf1[col] = pd.to_datetime(sf1[col], errors="coerce")

art = sf1[(sf1.dimension == "ART") & (sf1.ticker == "AAPL")].copy()
art = art[(art.datekey >= "2020-01-01") & (art.datekey <= "2021-12-31")].sort_values("datekey")
cols = ["ticker", "reportperiod", "datekey", "ncfo", "capex", "fcf"]
available = [c for c in cols if c in art.columns]
df = art[available].copy()
df["fcf_recon_a"] = df["ncfo"] - df["capex"]
df["fcf_recon_b"] = df["ncfo"] + df["capex"]
display(df)

# Compare to Sharadar fcf
if "fcf" in art.columns:
    match_a = np.isclose(df["fcf_recon_a"], df["fcf"], rtol=1e-5, equal_nan=True).sum()
    match_b = np.isclose(df["fcf_recon_b"], df["fcf"], rtol=1e-5, equal_nan=True).sum()
    n = len(df)
    print(f"Rows: {n}. fcf_recon_a (ncfo - capex) matches fcf: {match_a}. fcf_recon_b (ncfo + capex) matches fcf: {match_b}.")
    if match_b >= match_a and match_b > 0:
        print("Conclusion: Use fcf_recon = ncfo + capex (Sharadar capex is negative).")
    else:
        print("Conclusion: Use fcf_recon = ncfo - capex (Sharadar capex is positive).")
else:
    print("No 'fcf' column in SF1; check column names.")

,ticker,reportperiod,datekey,ncfo,capex,fcf,fcf_recon_a,fcf_recon_b
1815430,AAPL,2019-12-28,2020-01-29,7.321700e+10,-9.247000e+09,6.397000e+10,8.246400e+10,6.397000e+10
2298205,AAPL,2020-03-28,2020-05-01,7.537300e+10,-8.737000e+09,6.663600e+10,8.411000e+10,6.663600e+10
1815431,AAPL,2020-06-27,2020-07-31,8.000800e+10,-8.302000e+09,7.170600e+10,8.831000e+10,7.170600e+10
1815432,AAPL,2020-09-26,2020-10-30,8.067400e+10,-7.309000e+09,7.336500e+10,8.798300e+10,7.336500e+10
1815433,AAPL,2020-12-26,2021-01-28,8.892100e+10,-8.702000e+09,8.021900e+10,9.762300e+10,8.021900e+10
1819201,AAPL,2021-03-27,2021-04-29,9.959100e+10,-9.118000e+09,9.047300e+10,1.087090e+11,9.047300e+10
1819202,AAPL,2021-06-26,2021-07-28,1.044140e+11,-9.646000e+09,9.476800e+10,1.140600e+11,9.476800e+10
1819203,AAPL,2021-09-25,2021-10-29,1.040380e+11,-1.108500e+10,9.295300e+10,1.151230e+11,9.295300e+10


Rows: 8. fcf_recon_a (ncfo - capex) matches fcf: 0. fcf_recon_b (ncfo + capex) matches fcf: 8.
Conclusion: Use fcf_recon = ncfo + capex (Sharadar capex is negative).


## Verify Sharadar grossmargin (AAPL ARQ)

Before trusting `grossmargin_slope` we need to confirm whether Sharadar's **grossmargin** is a ratio (0-1, e.g. 0.38 for 38%) or gross profit in dollars. AAPL gross margin is roughly 38-42%; if the field is in billions it is dollars and we should use gp/revenue instead.

In [ ]:
# Grossmargin verification: load SF1 ARQ for AAPL 2019-2021
import sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import config

path = config.DATA_DIR / "SF1.parquet"
if not path.exists():
    path = config.DATA_DIR / "sf1.parquet"
df = pd.read_parquet(path)
df.columns = df.columns.str.lower()
df["datekey"] = pd.to_datetime(df["datekey"], errors="coerce")
# Check if gp exists for fallback
has_gp = "gp" in df.columns
cols = ["ticker", "reportperiod", "grossmargin"]
if has_gp:
    cols.extend(["gp", "revenue"])
arq = df[(df.dimension == "ARQ") & (df.ticker == "AAPL") & (df.datekey >= "2019-01-01") & (df.datekey <= "2021-12-31")]
arq = arq.sort_values("reportperiod")
if has_gp:
    arq = arq.copy()
    arq["gm_computed"] = arq["gp"] / arq["revenue"].replace(0, pd.NA)
result = arq[[c for c in cols + (["gm_computed"] if has_gp else []) if c in arq.columns]]
display(result)
# grossmargin should be ~0.38-0.42 for AAPL if ratio; if billions it is dollars
assert (result["grossmargin"].dropna() < 2).all(), "grossmargin appears to be dollars not ratio - use gp/revenue"
print("OK: grossmargin is in ratio form (0-1).")

,ticker,reportperiod,grossmargin,gp,revenue,gm_computed
1811264,AAPL,2018-12-29,0.380,3.203100e+10,8.431000e+10,0.379919
2296407,AAPL,2019-03-30,0.376,2.182100e+10,5.801500e+10,0.376127
1811265,AAPL,2019-06-29,0.376,2.022700e+10,5.380900e+10,0.375904
1811266,AAPL,2019-09-28,0.380,2.431300e+10,6.404000e+10,0.379653
1811267,AAPL,2019-12-28,0.384,3.521700e+10,9.181900e+10,0.383548
1811268,AAPL,2020-03-28,0.384,2.237000e+10,5.831300e+10,0.383619
1811270,AAPL,2020-06-27,0.380,2.268000e+10,5.968500e+10,0.379995
1811272,AAPL,2020-09-26,0.382,2.468900e+10,6.469800e+10,0.381604
1811273,AAPL,2020-12-26,0.398,4.432800e+10,1.114390e+11,0.397778
1811274,AAPL,2021-03-27,0.425,3.807900e+10,8.958400e+10,0.425065


OK: grossmargin is in ratio form (0-1).


---

## Run 02_fundamentals on small slice + spot-checks (optional)

*Separate from the capex sign check above.* Run 02_fundamentals first (e.g. from terminal: `python pipeline/02_fundamentals.py`), then run the cell below to validate the output on a small slice and spot-check metrics.

In [ ]:
# 2.2 + 2.3: Load fundamental_pit, filter to small slice, assert and spot-check
import sys
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import config

fp_path = config.FUNDAMENTAL_PIT_PATH
if not fp_path.exists():
    print("Run 02_fundamentals.py first so that", fp_path, "exists.")
else:
    pit = pd.read_parquet(fp_path)
    pit.columns = pit.columns.str.lower()
    # Filter to 1–2 tickers and short date range (e.g. AAPL, 2020)
    subset = pit[(pit.ticker == "AAPL") & (pit["date"].astype(str).str.startswith("2020"))].copy()
    subset = subset.sort_values(["ticker", "date"]).head(20)
    display(subset[["ticker", "date", "art_datekey", "arq_datekey", "datekey", "pe_pit", "pb_pit", "pcf_pit", "accrual_ratio", "capex_intensity", "fcf_recon_ttm"]].head(10))

    # Assert: datekey = GREATEST(art_datekey, arq_datekey) when both or one present
    for _, r in subset.iterrows():
        ak, qk = r.get("art_datekey"), r.get("arq_datekey")
        dk = r.get("datekey")
        if pd.notna(ak) or pd.notna(qk):
            expected = max(pd.Timestamp(ak) if pd.notna(ak) else pd.Timestamp("1900-01-01"),
                          pd.Timestamp(qk) if pd.notna(qk) else pd.Timestamp("1900-01-01"))
            if expected != pd.Timestamp("1900-01-01"):
                assert dk is not None, "datekey should be non-null"
                assert pd.Timestamp(dk) == expected, f"datekey {dk} != GREATEST(art,arq) {expected}"
    print("Assert: datekey = GREATEST(art_datekey, arq_datekey) — OK")

    # Spot-check one row: accrual_ratio, capex_intensity, fcf_recon_ttm vs raw ART
    if "sf1" not in dir() or sf1 is None or sf1.empty:
        sf1 = pd.read_parquet(config.DATA_DIR / "SF1.parquet" if (config.DATA_DIR / "SF1.parquet").exists() else config.DATA_DIR / "sf1.parquet")
        sf1.columns = sf1.columns.str.lower()
    art = sf1[sf1.dimension == "ART"].copy()
    art["datekey"] = pd.to_datetime(art["datekey"], errors="coerce")
    row = subset.iloc[0]
    t, d = row["ticker"], pd.Timestamp(row["date"])
    art_datekey = row.get("art_datekey")
    if pd.notna(art_datekey):
        raw = art[(art.ticker == t) & (art.datekey == pd.Timestamp(art_datekey))].iloc[0]
        rev = raw.get("revenueusd") or raw.get("revenue")
        accrual_manual = (raw["netinccmn"] - raw["ncfo"]) / raw["assets"] if raw.get("assets") and raw["assets"] != 0 else None
        capint_manual = abs(raw["capex"]) / rev if rev and rev != 0 else None
        fcf_manual = raw["ncfo"] + raw["capex"]
        print(f"Spot-check ({t}, {d.date()}): art_datekey={art_datekey}")
        print(f"  accrual_ratio: pit={row.get('accrual_ratio')}  manual={accrual_manual}")
        print(f"  capex_intensity: pit={row.get('capex_intensity')}  manual={capint_manual}")
        print(f"  fcf_recon_ttm: pit={row.get('fcf_recon_ttm')}  manual (ncfo+capex)={fcf_manual}")
    else:
        print("No art_datekey for first row; skip spot-check.")
    print("Done.")

In [ ]:
# Universe: check for duplicate (ticker, date) rows (e.g. from TICKERS having SF1 + SEP rows per ticker)
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import config

ticker_check = "AAPL"  # change to any ticker
path = config.DAILY_UNIVERSE_PATH
if not path.exists():
    print(f"Universe not found: {path}. Run pipeline 01_universe.py first.")
else:
    u = pd.read_parquet(path)
    sub = u[u["ticker"] == ticker_check].copy()
    if sub.empty:
        print(f"Ticker {ticker_check} not found in universe.")
    else:
        sub["date"] = pd.to_datetime(sub["date"]).dt.normalize()
        n_rows = len(sub)
        n_dates = sub["date"].nunique()
        dupes = sub[sub.duplicated(subset=["date"], keep=False)].sort_values("date")
        n_dupe_dates = dupes["date"].nunique() if len(dupes) > 0 else 0
        print(f"Ticker: {ticker_check}")
        print(f"  Total rows: {n_rows}, Unique dates: {n_dates}")
        print(f"  Duplicate dates: {n_dupe_dates} dates have >1 row")
        if n_dupe_dates > 0:
            print(f"\n  Sample of duplicate rows (first 10 dates with dupes):")
            sample_dates = dupes["date"].unique()[:10]
            display(dupes[dupes["date"].isin(sample_dates)])
        else:
            print("  No duplicates per date.")

Ticker: AAPL
  Total rows: 12578, Unique dates: 6289
  Duplicate dates: 6289 dates have >1 row

  Sample of duplicate rows (first 10 dates with dupes):


,ticker,date,in_universe,days_listed,fwd_delisted_30d,fwd_delisted_90d,fwd_acquired_90d,fwd_spinoff_60d,sector,famaindustry,scalemarketcap
43815932,AAPL,2000-01-03,True,5115,False,False,False,False,Technology,Computers,18050.0
13998430,AAPL,2000-01-03,True,5115,False,False,False,False,Technology,Computers,18050.0
25880777,AAPL,2000-01-04,True,5116,False,False,False,False,Technology,Computers,16516.0
36178262,AAPL,2000-01-04,True,5116,False,False,False,False,Technology,Computers,16516.0
45564672,AAPL,2000-01-05,True,5117,False,False,False,False,Technology,Computers,16750.0
42161944,AAPL,2000-01-05,True,5117,False,False,False,False,Technology,Computers,16750.0
60241418,AAPL,2000-01-06,True,5118,False,False,False,False,Technology,Computers,15306.0
26116218,AAPL,2000-01-06,True,5118,False,False,False,False,Technology,Computers,15306.0
67973119,AAPL,2000-01-07,True,5119,False,False,False,False,Technology,Computers,16028.0
43936516,AAPL,2000-01-07,True,5119,False,False,False,False,Technology,Computers,16028.0


In [ ]:
# TICKERS duplicate investigation: after filtering to table='SF1', which ticker still has 2 rows?
# Run this when 01_universe.py logs "Still have duplicates after SF1 filter - investigate"
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import config

path = config.DATA_DIR / "TICKERS.parquet"
if not path.exists():
    path = config.DATA_DIR / "tickers.parquet"
if not path.exists():
    print("TICKERS parquet not found.")
else:
    df = pd.read_parquet(path)
    df.columns = df.columns.str.lower()
    # Filter same as 01_universe: WHERE "table" = 'SF1'
    table_col = "table" if "table" in df.columns else None
    if table_col is None:
        print("No 'table' column in TICKERS. Columns:", list(df.columns))
    else:
        sf1 = df[df[table_col].astype(str).str.upper() == "SF1"].copy()
        n_rows = len(sf1)
        n_tickers = sf1["ticker"].nunique()
        print(f"TICKERS (SF1 only): {n_rows} rows, {n_tickers} unique tickers")
        if n_rows != n_tickers:
            dup_tickers = sf1.groupby("ticker").size()
            dup_tickers = dup_tickers[dup_tickers > 1].index.tolist()
            print(f"Tickers with >1 row: {dup_tickers}")
            if dup_tickers:
                dup = sf1[sf1["ticker"].isin(dup_tickers)].sort_values("ticker")
                display(dup)
                for t in dup_tickers:
                    rows = dup[dup["ticker"] == t]
                    diffs = [c for c in rows.columns if rows[c].nunique() > 1]
                    print(f"\n{t}: columns that differ between rows: {diffs}")
            else:
                # No ticker appears twice → extra row likely has null/empty ticker
                null_ticker = sf1[sf1["ticker"].isna()]
                if len(null_ticker) == 0:
                    null_ticker = sf1[sf1["ticker"].astype(str).str.strip() == ""]
                print("No ticker appears twice; discrepancy likely one row with null/empty ticker.")
                print(f"Rows with null/empty ticker: {len(null_ticker)}")
                if len(null_ticker) > 0:
                    display(null_ticker)
        else:
            print("No duplicates after SF1 filter.")

TICKERS (SF1 only): 17561 rows, 17560 unique tickers
Tickers with >1 row: []


,table,permaticker,ticker,name,exchange,isdelisted,category,cusips,siccode,sicsector,...,currency,location,lastupdated,firstadded,firstpricedate,lastpricedate,firstquarter,lastquarter,secfilings,companysite


,table,permaticker,ticker,name,exchange,isdelisted,category,cusips,siccode,sicsector,...,currency,location,lastupdated,firstadded,firstpricedate,lastpricedate,firstquarter,lastquarter,secfilings,companysite
10752,SF1,638812,None,NANO LABS LTD,NASDAQ,N,ADR Common Stock,G6391Y128 63011A102 G6391Y110,3674.0,Manufacturing,...,USD,China,2025-09-17,2022-07-12,2022-07-12,2026-02-20,2020-12-31,2025-06-30,https://www.sec.gov/cgi-bin/browse-edgar?actio...,https://www.nano.cn
